# Step 7e: do the 10 m classifiers recognise vessels, or the image source?

The 10 m classifiers of step 7c scored almost as high as the high-resolution ones, which is more than a vessel of a few pixels should allow. A possible reason: the fixed crop window is 56 pixels, and at 10 m a whole scene is only 28 to 60 pixels wide, so each crop shows the entire scene with a black border. That gives away which source dataset the image came from, and the sources differ in what they contain.

This compares the ResNet101 at each level with a rule that knows only the source and always answers with that source's most frequent class. If the classifier does not clearly beat the rule within each source, it has learned the source, not the vessel.

Reads the dataset straight from Drive, so it takes several minutes before the first table. A GPU helps a little; nothing is trained and nothing is written.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO_URL = 'https://github.com/bprtdaniel/vessel-watch.git'
DATA_DIR = '/content/drive/MyDrive/5.Projects/Data/ShipRSImageNet_V1'
RUNS_DIR = '/content/drive/MyDrive/5.Projects/runs'

In [ ]:
import os, subprocess

REPO_DIR = '/content/vessel-watch'
if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
%pip install -q --no-deps -e /content/vessel-watch
%cd /content/vessel-watch
!git log --oneline -1

os.environ['VESSELWATCH_DATA'] = DATA_DIR
os.environ['VESSELWATCH_RUNS'] = RUNS_DIR
os.environ['VESSELWATCH_CACHE'] = '/content/cache'

In [ ]:
for level in (1, 2, 3):
    !python -m vesselwatch.by_source --config configs/crops_10m/level{level}_resnet101.yaml --weights "$RUNS_DIR/crops10m_level{level}_resnet101/best.pt" --num-workers 2